# Hướng Dẫn Tổ Chức Dịch Vụ
**Lưu ý:** file này chỉ được sử dụng để tổ chức dịch vụ LLM trên colab và được tunneling ra bên ngoài thông qua Ngrok  

- Chạy các ô bên dưới một cách tuần tự để thu được 1 link tunneling ra bên ngoài  

- Dán nó vào trường OLLAMA_BASE_URL trong file danger-service.py hoặc để nó trong file .env và gọi sử dụng file  

- Nếu như cầu sử dụng thay đổi( ví dụ như thay đổi model) thì nên thay đổi tên model sử dụng trong file này khi tải về và tên model trong danger_service.py  

- Sau khi hoàn thành thì chạy trên terminal lệnh  ```ollama```  

- Chọn model và chạy model đã chọn từ đầu

**Lưu ý**: Nếu dịch vụ chạy không thành công( lệnh curl ở ô dưới không có kết quả như mong đợi) thì hãy sử dụng các lệnh nằm sau dấu '!' trong phần code và sử dụng nó trong terminal của colab
    Nên bị lỗi 403 forbiden cân nhắc đóng tunneling và thử khởi tạo tunnelin thông qua lệnh được để ở bên dưới cùng

# Tải Model cần thiết về

In [ ]:
# 1. Cài đặt zstd để giải nén Ollama
!apt-get install -y zstd
!pip install pyngrok
# 2. Cài đặt Ollama
!curl -fsSL https://ollama.com/install.sh | sh

import time
time.sleep(30) # Đợi 5 giây để server khởi động hoàn toàn

# 4. Tải model (Lưu ý: Kiểm tra tên model chính xác, ví dụ qwen2.5:3b hoặc qwen2.5:7b)
# Hiện tại dòng Qwen phổ biến là 2.5, bạn nên kiểm tra lại phiên bản 3.5
!ollama pull gemma4:e2b
print("✅ Ollama đã sẵn sàng!")

# Khởi Động Dịch Vụ

In [ ]:
import os
import subprocess
import time
import requests

# 1. Set env vars
os.environ['OLLAMA_HOST'] = '0.0.0.0'
os.environ['OLLAMA_ORIGINS'] = '*'

# 2. Khởi động Ollama
subprocess.Popen(['ollama', 'serve'],
                 env={**os.environ, 'OLLAMA_HOST': '0.0.0.0', 'OLLAMA_ORIGINS': '*'})

# 3. Chờ Ollama sẵn sàng (poll thay vì sleep cố định)
print("Chờ Ollama khởi động...")
for i in range(30):
    try:
        r = requests.get('http://localhost:11434')
        if r.status_code == 200:
            print(f"✅ Ollama sẵn sàng sau {i+1}s")
            break
    except:
        time.sleep(1)
else:
    print("❌ Ollama không khởi động được")

# Tunneling Thông Qua Ngrok

In [ ]:
import os
from pyngrok import ngrok

# --- CẤU HÌNH XÁC THỰC ---
NGROK_TOKEN = "NGROK_TUNNELING_KEY"
ngrok.set_auth_token(NGROK_TOKEN)

# Khởi động Ollama server
get_ipython().system_raw('ollama serve &')

# --- MỞ CỔNG RA NGOÀI ---
# Kết nối tunnel tới cổng 11434 của Ollama
public_url = ngrok.connect(11434, "http")

print(f"🚀 Đường truyền đã thông suốt!")
print(f"🔗 URL công khai của bạn: {public_url.public_url}")

# Kiểm Thử

In [ ]:
!curl http://127.0.0.1:11434/api/tags

In [ ]:
!curl http://localhost:11434

# Tắt Ngrok tunneling
**Chỉ sử dụng khi không thể tunneling bằng link cũ**

In [ ]:
# 1. Tắt toàn bộ tiến trình Ollama và ngrok cũ để tránh xung đột
!pkill ollama
ngrok.kill()